# 🎓 Agente de IA (RAG) — Instituto Educativo Horizonte

Este notebook construye, paso a paso, un agente de inteligencia artificial con **RAG**
(Retrieval-Augmented Generation) que responde preguntas sobre **6 documentos del
instituto**: el manual académico, el reglamento del estudiante, la política de
reembolsos, las preguntas frecuentes de cursos y certificados, la guía de la plataforma,
y el programa de becas y afiliados.

## ¿Qué es RAG y cómo lo implementamos aquí?

RAG significa que, en vez de "meterle" todo el contenido al modelo de lenguaje de una vez,
primero **buscamos** (Retrieval) los fragmentos más relevantes para la pregunta, y solo
esos fragmentos se le pasan al modelo para que redacte la respuesta (Generation).

> **Nota de diseño importante:** al principio de este proyecto probamos fragmentos
> pequeños (500-800 caracteres), pero eso partía tablas completas (como el calendario o
> los proyectos por materia) a la mitad, y el agente daba respuestas incompletas según
> cómo estuviera redactada la pregunta. Como los 6 documentos son cortos, aquí usamos
> fragmentos grandes (4000 caracteres): cada documento completo casi siempre cabe en un
> solo fragmento. Sigue siendo RAG real (hay búsqueda por similitud), solo que con una
> granularidad más gruesa — y mucho más confiable.

## Cómo se usa este notebook

- El notebook está dividido en celdas. Unas son de **texto** (como esta) y otras son de
  **código** (con fondo gris y un botón ▶️ a la izquierda).
- Debes ejecutar las celdas de código **en orden, de arriba hacia abajo**, sin saltarte ninguna.
- Para ejecutar una celda: haz clic en el botón ▶️ que aparece a la izquierda de la celda,
  o selecciónala y presiona `Shift + Enter`.
- Mientras una celda se está ejecutando verás un ícono girando ⏳ o un asterisco `[*]` a la
  izquierda. **Espera a que termine** (verás un número, por ejemplo `[3]`) antes de
  ejecutar la siguiente.
- Si algo falla, casi siempre se soluciona volviendo a ejecutar desde el Paso 1 (a veces
  Colab "reinicia" el entorno y hay que reinstalar todo).

Antes de empezar, ten a la mano los 6 PDFs guardados en tu computadora:
1. `manual_academico_instituto_horizonte.pdf`
2. `reglamento_del_estudiante.pdf`
3. `politica_reembolso_matriculas.pdf`
4. `faq_cursos_certificados.pdf`
5. `guia_uso_plataforma.pdf`
6. `programa_becas_afiliados.pdf`

Y una clave de API gratuita de Cohere (te explico cómo conseguirla en el Paso 6).

## Paso 1 — Instalar las herramientas necesarias

Esto solo se hace una vez por sesión. Instalamos:
- **LangChain / langchain-community**: para leer los PDFs.
- **langchain-classic**: desde LangChain 1.0, `RetrievalQA` vive en este paquete aparte.
- **langchain-text-splitters**: para dividir los documentos en fragmentos.
- **langchain-cohere**: para conectar el modelo de lenguaje de Cohere.
- **PyPDF**: el lector de PDFs.
- **FAISS**: la base de datos donde guardamos los fragmentos para buscar por significado.
- **sentence-transformers**: convierte el texto en vectores (embeddings), gratis y local.
- **cohere**: el cliente del modelo de lenguaje.

Ejecuta la siguiente celda y espera a que termine (puede tardar 1-2 minutos, es normal).

Si al ejecutar la celda, te sale el siguiente error:
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
No te preocupes es una advertencia, de compatibilidad de librerías de Python, para este caso específico no tiene afectación.


In [1]:
!pip install -q langchain langchain-classic langchain-community langchain-cohere \
    langchain-text-splitters pypdf faiss-cpu sentence-transformers cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Paso 2 — Subir los PDFs del instituto

Al ejecutar la celda de abajo aparecerá un botón que dice **"Choose Files"** (Elegir archivos).

1. Haz clic en ese botón.
2. Selecciona **los 6 PDFs a la vez** (mantén presionada `Ctrl` en Windows o `Cmd` en Mac
   mientras haces clic en cada archivo, para seleccionarlos todos juntos).
3. Espera a que se carguen al 100% en cada uno y

In [2]:
from google.colab import files
subido = files.upload()

Saving faq_cursos_certificados.pdf to faq_cursos_certificados.pdf
Saving guia_uso_plataforma.pdf to guia_uso_plataforma.pdf
Saving manual_academico_instituto_horizonte.pdf to manual_academico_instituto_horizonte.pdf
Saving politica_reembolso_matriculas.pdf to politica_reembolso_matriculas.pdf
Saving programa_becas_afiliados.pdf to programa_becas_afiliados.pdf
Saving reglamento_del_estudiante.pdf to reglamento_del_estudiante.pdf


## Paso 3 — Leer los PDFs y unir sus páginas

Esta celda abre cada uno de los PDFs y une todas sus páginas en un solo
texto por documento. Esto es importante: si dejáramos cada página como un fragmento
separado, una tabla que empiece en una página y termine en la siguiente podría quedar
partida a la mitad, sin que el tamaño de fragmento (`chunk_size`) del Paso 4 pueda
evitarlo.

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

archivos_pdf = list(subido.keys())  # usa automáticamente los nombres de los archivos que subiste

paginas = []
for archivo in archivos_pdf:
    loader = PyPDFLoader(archivo)
    paginas_pdf = loader.load()
    texto_completo = "\n".join(p.page_content for p in paginas_pdf)
    paginas.append(Document(page_content=texto_completo, metadata={"source": archivo}))

print(f"✅ Documentos cargados y unidos: {len(archivos_pdf)} -> {archivos_pdf}")
print(f"✅ Tamaño de cada documento (caracteres): {[len(p.page_content) for p in paginas]}")
print("\nVista previa del primer documento:\n")
print(paginas[0].page_content[:500])

/tmp/ipykernel_4412/1309463463.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ Documentos cargados y unidos: 6 -> ['faq_cursos_certificados.pdf', 'guia_uso_plataforma.pdf', 'manual_academico_instituto_horizonte.pdf', 'politica_reembolso_matriculas.pdf', 'programa_becas_afiliados.pdf', 'reglamento_del_estudiante.pdf']
✅ Tamaño de cada documento (caracteres): [2829, 2990, 5109, 2729, 3251, 3862]

Vista previa del primer documento:

Instituto Educativo Horizonte
 Preguntas Frecuentes sobre Cursos y Certificados
 Documento de referencia para estudiantes
Este documento reúne las preguntas más frecuentes que hacen los estudiantes sobre la duración de los
cursos, los requisitos para obtener un certificado, y el procedimiento en caso de pérdida o reposición de
constancias.
Sobre los cursos
¿Cuánto dura cada materia?
Todas las materias del plan de estudios tienen duración semestral, con clases de agosto a diciembre o de
enero a j


## Paso 4 — Dividir los documentos en fragmentos

Un modelo de lenguaje no puede "leer" muchos PDFs completos de un solo golpe de forma
eficiente en cada pregunta, así que dividimos el contenido en fragmentos que luego se
buscan por similitud.

> Usamos `chunk_size=4000` a propósito: como cada documento es corto, esto hace que cada
> uno quede casi siempre en un solo fragmento completo (una tabla o una lista nunca queda
> partida a la mitad).

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=300,
)
fragmentos = splitter.split_documents(paginas)
print(f"✅ Fragmentos generados: {len(fragmentos)}")

✅ Fragmentos generados: 7


## Paso 5 — Convertir los fragmentos en vectores y crear la base de búsqueda

Cada fragmento se convierte en un "embedding" (una lista de números que representa su
significado), para poder encontrar rápidamente los más relacionados con una pregunta.

Este paso puede tardar uno o dos minutos la primera vez, porque descarga el modelo que
genera los embeddings.

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
base_vectorial = FAISS.from_documents(fragmentos, embeddings)

print("✅ Base de búsqueda creada correctamente")

/tmp/ipykernel_4412/1398060328.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Base de búsqueda creada correctamente


## Paso 6 — Colocar tu clave de API (Cohere)

Al ejecutar la celda de abajo aparecerá un recuadro para pegar tu clave, una vez que la ingreses da Enter. El texto que
escribas ahí **no se mostrará en pantalla** (por seguridad), es normal.

In [6]:
import os
from getpass import getpass

os.environ["COHERE_API_KEY"] = getpass("Pega aquí tu clave de API de Cohere y presiona Enter: ")
print("✅ Clave guardada para esta sesión")

Pega aquí tu clave de API de Cohere y presiona Enter: ··········
✅ Clave guardada para esta sesión


## Paso 7 — Conectar el modelo de lenguaje

> Nota: Cohere retira modelos antiguos de vez en cuando. Si en el futuro ves un error
> `NotFoundError` diciendo que un modelo "fue removido", revisa la lista de modelos
> vigentes en https://docs.cohere.com/docs/models y cambia el nombre en la celda de abajo.

In [7]:
from langchain_cohere import ChatCohere

llm = ChatCohere(model="command-a-03-2025", temperature=0)
print("✅ Modelo de lenguaje listo")

✅ Modelo de lenguaje listo


## Paso 8 — Construir el agente (cadena de preguntas y respuestas)

Esta es la pieza que une todo: cuando alguien pregunta algo, el agente...
1. Busca en la base vectorial los fragmentos más parecidos a la pregunta (usamos `k=8`,
   generoso a propósito: como hay pocos fragmentos grandes en total, esto cubre
   prácticamente todos los documentos relevantes para cualquier pregunta).
2. Le pasa esos fragmentos + la pregunta al modelo de lenguaje, con una **instrucción
   personalizada** que le exige responder de forma completa (por ejemplo, no omitir
   materias, fechas o tipos de beca si la pregunta involucra una lista).
3. El modelo redacta la respuesta usando solo esa información.



In [8]:
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

retriever = base_vectorial.as_retriever(search_kwargs={"k": 8})

plantilla_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Eres el agente académico del Instituto Educativo Horizonte. "
        "Responde la pregunta del estudiante usando SOLO la información del "
        "siguiente contexto. Si la respuesta involucra una lista o tabla "
        "(por ejemplo, materias, fechas, tipos de beca o periodos de examen), "
        "incluye TODOS los elementos relevantes que aparezcan en el contexto, "
        "no solo algunos. Antes de responder, revisa TODO el contexto en busca de "
        "cualquier sección relacionada con el tema, aunque no forme parte de la "
        "misma lista o tabla (por ejemplo, si preguntan sobre proyectos a entregar, "
        "considera tanto los proyectos por materia como el proyecto final "
        "institucional, si ambos aparecen). Si la información no está en el "
        "contexto, dilo claramente.\n\n"
        "Contexto:\n{context}\n\n"
        "Pregunta: {question}\n"
        "Respuesta completa:"
    ),
)

agente = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": plantilla_prompt},
)
print("✅ Agente listo para responder preguntas")

✅ Agente listo para responder preguntas


## Paso 9 — ¡Probar el agente! Preguntas de ejemplo

Al ejecutar la celda el agente responderá preguntas reales sobre los documentos
del instituto.

In [9]:
preguntas_ejemplo = [
    "¿Cuándo es la entrega del proyecto final y qué características debe tener?",
    "¿Quién imparte Química II y en qué horario da asesorías?",
    "¿Qué proyectos tengo que entregar este semestre, de qué materias y cuándo?",
    "¿Cuándo son los periodos de exámenes parciales y finales?",
    "¿Qué tipos de becas ofrece el instituto y cuáles son los requisitos?",
    "¿Cómo entrego un proyecto en el Portal Académico?",
]

for p in preguntas_ejemplo:
    resultado = agente.invoke({"query": p})
    print("❓ PREGUNTA:", p)
    print("💬 RESPUESTA:", resultado["result"])
    print("-" * 80)

❓ PREGUNTA: ¿Cuándo es la entrega del proyecto final y qué características debe tener?
💬 RESPUESTA: La entrega del proyecto final institucional está programada para el **viernes 4 de diciembre de 2026**, antes de las 23:59 horas, y debe realizarse a través del **Portal Académico** en la sección "Entrega de proyecto final". Este proyecto debe cumplir con las siguientes características:

1. **Integración de conocimientos**: Debe relacionar conocimientos de al menos dos materias cursadas en el semestre.  
2. **Formato**: Se entrega en formato digital (PDF).  
3. **Estructura**: Debe incluir:  
   - Planteamiento del problema.  
   - Marco teórico.  
   - Metodología.  
   - Resultados.  
   - Conclusiones.  
4. **Extensión**: Mínimo 15 páginas.  
5. **Presentación oral**: Incluye una exposición de 15 minutos ante un panel de al menos dos profesores.  
6. **Modalidad**: Puede realizarse de forma individual o en equipos de hasta 3 estudiantes.  

Además, es importante considerar que este pr

## Paso 10 — Haz tu propia pregunta

Escribe cualquier pregunta sobre los documentos del instituto y presiona Enter.

In [11]:
mi_pregunta = input("Escribe tu pregunta: ")
resultado = agente.invoke({"query": mi_pregunta})
print("\n💬 RESPUESTA:", resultado["result"])

Escribe tu pregunta: quién imparte literatura

💬 RESPUESTA: La materia de Literatura es impartida por la profesora **Daniela Torres**. Su correo electrónico es **dtorres@ieh.edu.mx** y su horario de asesoría es los **jueves de 11:00 a 12:30** en el **Cubículo 2**.
